## Stage 1: Creating LexBERT

In [3]:
import torch
import torch.nn as nn
from transformers import BertModel

class LexBertRanker(nn.Module):
    def __init__(self, bert_model_name="bert-base-uncased", n_lex_features=5):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        hidden = self.bert.config.hidden_size  # 768

        #NEW LAYER — learns to weight lexical signals
        self.lex_projection = nn.Linear(n_lex_features, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)

        # Fusion: 768 (BERT) + 64 (lex) → 1 score
        self.classifier = nn.Linear(hidden + 64, 1)

    def forward(self, input_ids, attention_mask, token_type_ids, lex_features):
        bert_out = self.bert(input_ids=input_ids,
                             attention_mask=attention_mask,
                             token_type_ids=token_type_ids)
        cls_vec = bert_out.last_hidden_state[:, 0, :]          # (B, 768)
        lex_vec = self.dropout(self.relu(self.lex_projection(lex_features)))  # (B, 64)
        combined = torch.cat([cls_vec, lex_vec], dim=1)        # (B, 832)
        return self.classifier(combined).squeeze(-1)            # (B,)

## Stage 2 : Wikipedia DAPT


In [4]:
from datasets import load_dataset
from transformers import (BertForMaskedLM, BertTokenizer,
                          DataCollatorForLanguageModeling,
                          Trainer, TrainingArguments)

# THIS is how you get Wikipedia
wiki = load_dataset("wikimedia/wikipedia", "20231101.en", split="train[:5%]")

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
mlm_model  = BertForMaskedLM.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512,
                     padding="max_length", return_special_tokens_mask=True)

tokenized = wiki.map(tokenize, batched=True,
                     remove_columns=["text","title","id","url"])

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

trainer = Trainer(
    model=mlm_model,
    args=TrainingArguments(
        output_dir="./bert-wiki-dapt",
        num_train_epochs=1,
        per_device_train_batch_size=16,
        learning_rate=2e-5,
        fp16=True,
        logging_steps=500,
        save_steps=5000,
    ),
    train_dataset=tokenized,
    data_collator=collator,
)
trainer.train()

# Save only the BERT backbone — you load this into LexBertRanker next
mlm_model.bert.save_pretrained("./bert-wiki-backbone")
tokenizer.save_pretrained("./bert-wiki-backbone")

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 6392.55it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss


KeyboardInterrupt: 

In [1]:
import torch
from transformers import BertForMaskedLM, BertTokenizer

# Load from the latest checkpoint that completed
mlm_model = BertForMaskedLM.from_pretrained("./bert-wiki-dapt/checkpoint-15000")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Save just the BERT backbone
mlm_model.bert.save_pretrained("./bert-wiki-backbone")
tokenizer.save_pretrained("./bert-wiki-backbone")
print("Backbone saved to ./bert-wiki-backbone")

c:\Users\argon\Ajinkya\nlp_proj_final\lexbert_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.80it/s]

Backbone saved to ./bert-wiki-backbone


In [5]:
import shutil, os

# use checkpoint-15000 — it's the most trained
src = "./bert-wiki-dapt/checkpoint-15000"
dst = "./bert-wiki-backbone"

os.makedirs(dst, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True)
print("Done — backbone ready at ./bert-wiki-backbone")
print(os.listdir(dst))  # confirm files are there

Done — backbone ready at ./bert-wiki-backbone
['config.json', 'model.safetensors', 'optimizer.pt', 'rng_state.pth', 'scaler.pt', 'scheduler.pt', 'tokenizer.json', 'tokenizer_config.json', 'trainer_state.json', 'training_args.bin']


## Stage 3 : Ranking pre-train on Natural Questions

This is where the lex layer actually learns to use lexical features:

In [7]:
import gc
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")
print(f"Using device: {device}")

# --- model ---
class LexBertRanker(nn.Module):
    def __init__(self, bert_model_name="bert-base-uncased", n_lex_features=5):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        hidden = self.bert.config.hidden_size  # 768

        self.lex_projection  = nn.Linear(n_lex_features, 64)
        self.relu            = nn.ReLU()
        self.dropout         = nn.Dropout(0.1)
        self.q_proj          = nn.Linear(hidden, 256)
        self.p_proj          = nn.Linear(hidden, 256)
        self.lex_classifier  = nn.Linear(256 + 64, 1)

    def encode(self, input_ids, attention_mask, token_type_ids):
        out = self.bert(input_ids=input_ids,
                        attention_mask=attention_mask,
                        token_type_ids=token_type_ids)
        return out.last_hidden_state[:, 0, :]  # (B, 768)

    def project_query(self, cls_vec):
        return self.q_proj(cls_vec) / 16.0     # (B, 256) with temperature

    def project_passage(self, cls_vec):
        return self.p_proj(cls_vec)             # (B, 256)

    def lex_score(self, q_proj, lex_features):
        lex_vec  = self.dropout(self.relu(self.lex_projection(lex_features)))
        combined = torch.cat([q_proj, lex_vec], dim=1)
        return self.lex_classifier(combined).squeeze(-1)  # (B,)

    def forward(self, q_input_ids, q_attention_mask, q_token_type_ids,
                p_input_ids, p_attention_mask, p_token_type_ids, lex_features):
        q_cls  = self.encode(q_input_ids, q_attention_mask, q_token_type_ids)
        p_cls  = self.encode(p_input_ids, p_attention_mask, p_token_type_ids)
        q_proj = self.project_query(q_cls)
        p_proj = self.project_passage(p_cls)
        dot    = (q_proj * p_proj).sum(dim=-1)
        lex    = self.lex_score(q_proj, lex_features)
        return dot + lex

# --- lexical features ---
def compute_lex_features(query, passage):
    qt = query.lower().split()
    pt = passage.lower().split()
    qs, ps = set(qt), set(pt)

    overlap  = len(qs & ps) / len(qs) if qs else 0.0
    jaccard  = len(qs & ps) / len(qs | ps) if (qs | ps) else 0.0
    try:
        mat = TfidfVectorizer().fit_transform([query, passage])
        tfidf_sim = cosine_similarity(mat[0], mat[1])[0][0]
    except:
        tfidf_sim = 0.0
    length  = np.log1p(len(pt)) / 10.0
    density = sum(pt.count(t) for t in qt) / max(len(pt), 1)

    return np.array([overlap, jaccard, tfidf_sim, length, density], dtype=np.float32)

# --- dataset ---
class TripletDataset(Dataset):
    def __init__(self, hf_data, tokenizer, max_len=128):
        self.data    = hf_data
        self.tok     = tokenizer
        self.max_len = max_len

        cols = hf_data.column_names
        print(f"Dataset columns: {cols}")

        self.q_col   = "query"    if "query"    in cols else "question"
        self.pos_col = "positive" if "positive" in cols else (
                       "answer"   if "answer"   in cols else cols[1])
        print(f"Using: query='{self.q_col}' | positive='{self.pos_col}'")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        q   = row[self.q_col]
        pos = row[self.pos_col]

        qe = self.tok(q,   max_length=64,          padding="max_length",
                      truncation=True, return_tensors="pt")
        pe = self.tok(pos, max_length=self.max_len, padding="max_length",
                      truncation=True, return_tensors="pt")

        return {
            "q_input_ids":        qe["input_ids"].squeeze(0),
            "q_attention_mask":   qe["attention_mask"].squeeze(0),
            "q_token_type_ids":   qe["token_type_ids"].squeeze(0),
            "pos_input_ids":      pe["input_ids"].squeeze(0),
            "pos_attention_mask": pe["attention_mask"].squeeze(0),
            "pos_token_type_ids": pe["token_type_ids"].squeeze(0),
            "pos_lex":            torch.tensor(compute_lex_features(q, pos)),
        }

# --- cross-entropy in-batch loss (replaces margin loss) ---
def in_batch_negative_loss(model, batch, device):
    q_cls = model.encode(
        batch["q_input_ids"].to(device),
        batch["q_attention_mask"].to(device),
        batch["q_token_type_ids"].to(device),
    )
    p_cls = model.encode(
        batch["pos_input_ids"].to(device),
        batch["pos_attention_mask"].to(device),
        batch["pos_token_type_ids"].to(device),
    )

    q_proj = model.project_query(q_cls)    # (B, 256)
    p_proj = model.project_passage(p_cls)  # (B, 256)
    lex    = batch["pos_lex"].to(device)   # (B, 5)

    score_matrix = torch.matmul(q_proj, p_proj.T)          # (B, B)
    lex_bonus    = model.lex_score(q_proj, lex)             # (B,)
    score_matrix = score_matrix + torch.diag(lex_bonus)     # add lex to diagonal only

    # label i = i means query i should match passage i
    targets = torch.arange(score_matrix.size(0), device=device)
    return torch.nn.functional.cross_entropy(score_matrix, targets)

# --- data ---
print("Loading Natural Questions...")
nq = load_dataset("sentence-transformers/natural-questions", split="train[:20%]")
print(f"Loaded {len(nq)} training examples")

backbone  = "./bert-wiki-backbone" if os.path.exists("./bert-wiki-backbone") else "bert-base-uncased"
print(f"Loading from backbone: {backbone}")

tokenizer = BertTokenizer.from_pretrained(backbone)
model     = LexBertRanker(bert_model_name=backbone, n_lex_features=5).to(device)

optimizer = optim.AdamW([
    {"params": model.bert.parameters(),           "lr": 2e-5},
    {"params": model.q_proj.parameters(),         "lr": 1e-4},
    {"params": model.p_proj.parameters(),         "lr": 1e-4},
    {"params": model.lex_projection.parameters(), "lr": 1e-4},
    {"params": model.lex_classifier.parameters(), "lr": 1e-4},
], weight_decay=0.01)

loader = DataLoader(
    TripletDataset(nq, tokenizer),
    batch_size=16,
    shuffle=True,
    num_workers=0
)

# --- training ---
print("Starting Stage 3 training...")

for epoch in range(1):
    model.train()
    total_loss = 0

    for step, batch in enumerate(loader):
        optimizer.zero_grad()

        loss = in_batch_negative_loss(model, batch, device)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        if step % 100 == 0:
            print(f"  Step {step}/{len(loader)} | Loss: {loss.item():.4f}")

    print(f"Epoch 1 complete | Avg loss: {total_loss / len(loader):.4f}")

torch.save(model.state_dict(), "lex_bert_nq.pt")
print("Stage 3 complete — model saved to lex_bert_nq.pt")

CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Memory free: 2.54 GB
Using device: cuda
Loading Natural Questions...
Loaded 20046 training examples
Loading from backbone: ./bert-wiki-backbone


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 9078.58it/s]
[transformers] BertModel LOAD REPORT from: ./bert-wiki-backbone
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Dataset columns: ['query', 'answer']
Using: query='query' | positive='answer'
Starting Stage 3 training...
  Step 0/1253 | Loss: 2.7506
  Step 100/1253 | Loss: 0.8228
  Step 200/1253 | Loss: 0.3493
  Step 300/1253 | Loss: 0.1891
  Step 400/1253 | Loss: 0.0364
  Step 500/1253 | Loss: 0.0072
  Step 600/1253 | Loss: 0.0043
  Step 700/1253 | Loss: 0.0025
  Step 800/1253 | Loss: 0.0022
  Step 900/1253 | Loss: 0.0041
  Step 1000/1253 | Loss: 0.0010
  Step 1100/1253 | Loss: 0.0010
  Step 1200/1253 | Loss: 0.0009
Epoch 1 complete | Avg loss: 0.2282
Stage 3 complete — model saved to lex_bert_nq.pt


## Stage 4 : Fine-tune on MS MARCO

In [4]:
import gc
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

# --- model ---
class LexBertRanker(nn.Module):
    def __init__(self, bert_model_name="bert-base-uncased", n_lex_features=5):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        hidden = self.bert.config.hidden_size

        self.lex_projection = nn.Linear(n_lex_features, 64)
        self.relu           = nn.ReLU()
        self.dropout        = nn.Dropout(0.1)
        self.q_proj         = nn.Linear(hidden, 256)
        self.p_proj         = nn.Linear(hidden, 256)
        self.lex_classifier = nn.Linear(256 + 64, 1)

    def encode(self, input_ids, attention_mask, token_type_ids):
        out = self.bert(input_ids=input_ids,
                        attention_mask=attention_mask,
                        token_type_ids=token_type_ids)
        return out.last_hidden_state[:, 0, :]

    def project_query(self, cls_vec):
        # NO division here — temperature in loss handles scaling
        return self.q_proj(cls_vec)

    def project_passage(self, cls_vec):
        return self.p_proj(cls_vec)

    def lex_score(self, q_proj, lex_features):
        lex_vec  = self.dropout(self.relu(self.lex_projection(lex_features)))
        combined = torch.cat([q_proj, lex_vec], dim=1)
        return self.lex_classifier(combined).squeeze(-1)

    def forward(self, q_input_ids, q_attention_mask, q_token_type_ids,
                p_input_ids, p_attention_mask, p_token_type_ids, lex_features):
        q_cls  = self.encode(q_input_ids, q_attention_mask, q_token_type_ids)
        p_cls  = self.encode(p_input_ids, p_attention_mask, p_token_type_ids)
        q_proj = self.project_query(q_cls)
        p_proj = self.project_passage(p_cls)
        dot    = (q_proj * p_proj).sum(dim=-1)
        lex    = self.lex_score(q_proj, lex_features)
        return dot + lex

# --- lexical features ---
def compute_lex_features(query, passage):
    qt = query.lower().split()
    pt = passage.lower().split()
    qs, ps = set(qt), set(pt)

    overlap  = len(qs & ps) / len(qs) if qs else 0.0
    jaccard  = len(qs & ps) / len(qs | ps) if (qs | ps) else 0.0
    try:
        mat = TfidfVectorizer().fit_transform([query, passage])
        tfidf_sim = cosine_similarity(mat[0], mat[1])[0][0]
    except:
        tfidf_sim = 0.0
    length  = np.log1p(len(pt)) / 10.0
    density = sum(pt.count(t) for t in qt) / max(len(pt), 1)

    return np.array([overlap, jaccard, tfidf_sim, length, density], dtype=np.float32)

# --- dataset ---
class MarcoRankingDataset(Dataset):
    def __init__(self, hf_data, tokenizer, max_q=64, max_p=128, max_examples=200000):
        self.tok   = tokenizer
        self.max_q = max_q
        self.max_p = max_p
        self.pairs = []

        print("Building pairwise triplets from MS MARCO...")
        skipped = 0
        for ex in hf_data:
            if len(self.pairs) >= max_examples:
                break
            q        = ex["query"]
            texts    = ex["passages"]["passage_text"]
            selected = ex["passages"]["is_selected"]
            pos_passages = [t for t, s in zip(texts, selected) if s == 1]
            neg_passages = [t for t, s in zip(texts, selected) if s == 0]
            if not pos_passages or not neg_passages:
                skipped += 1
                continue
            for pos in pos_passages:
                for neg in neg_passages[:3]:
                    self.pairs.append((q, pos, neg))
                    if len(self.pairs) >= max_examples:
                        break
                if len(self.pairs) >= max_examples:
                    break

        print(f"Built {len(self.pairs)} triplets | Skipped {skipped} queries with no pos/neg")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        q, pos, neg = self.pairs[idx]
        qe = self.tok(q,   max_length=self.max_q, padding="max_length",
                      truncation=True, return_tensors="pt")
        pe = self.tok(pos, max_length=self.max_p, padding="max_length",
                      truncation=True, return_tensors="pt")
        ne = self.tok(neg, max_length=self.max_p, padding="max_length",
                      truncation=True, return_tensors="pt")
        return {
            "q_input_ids":        qe["input_ids"].squeeze(0),
            "q_attention_mask":   qe["attention_mask"].squeeze(0),
            "q_token_type_ids":   qe["token_type_ids"].squeeze(0),
            "pos_input_ids":      pe["input_ids"].squeeze(0),
            "pos_attention_mask": pe["attention_mask"].squeeze(0),
            "pos_token_type_ids": pe["token_type_ids"].squeeze(0),
            "neg_input_ids":      ne["input_ids"].squeeze(0),
            "neg_attention_mask": ne["attention_mask"].squeeze(0),
            "neg_token_type_ids": ne["token_type_ids"].squeeze(0),
            "pos_lex":            torch.tensor(compute_lex_features(q, pos)),
            "neg_lex":            torch.tensor(compute_lex_features(q, neg)),
        }

# --- ranking loss ---
def ranking_loss(model, batch, device, temperature=0.05):
    q_cls = model.encode(
        batch["q_input_ids"].to(device),
        batch["q_attention_mask"].to(device),
        batch["q_token_type_ids"].to(device),
    )
    p_cls = model.encode(
        batch["pos_input_ids"].to(device),
        batch["pos_attention_mask"].to(device),
        batch["pos_token_type_ids"].to(device),
    )
    n_cls = model.encode(
        batch["neg_input_ids"].to(device),
        batch["neg_attention_mask"].to(device),
        batch["neg_token_type_ids"].to(device),
    )

    q_proj  = model.project_query(q_cls)
    p_proj  = model.project_passage(p_cls)
    n_proj  = model.project_passage(n_cls)
    pos_lex = batch["pos_lex"].to(device)
    neg_lex = batch["neg_lex"].to(device)

    # normalize to unit vectors before dot product
    # this makes dot products bounded in [-1, 1]
    # so temperature=0.05 gives logits in [-20, 20] — proper range for CE
    q_proj = torch.nn.functional.normalize(q_proj, dim=-1)
    p_proj = torch.nn.functional.normalize(p_proj, dim=-1)
    n_proj = torch.nn.functional.normalize(n_proj, dim=-1)

    # (B, 2B) score matrix
    all_p        = torch.cat([p_proj, n_proj], dim=0)
    score_matrix = torch.matmul(q_proj, all_p.T)   # values in [-1, 1]

    # add lex bonus to diagonal
    lex_bonus = model.lex_score(q_proj, pos_lex)
    B         = q_proj.size(0)
    diag      = torch.arange(B, device=device)
    score_matrix[diag, diag] += lex_bonus

    # temperature scaling → logits in meaningful range
    score_matrix = score_matrix / temperature

    targets = torch.arange(B, device=device)
    ce_loss = torch.nn.functional.cross_entropy(score_matrix, targets)

    # pairwise margin loss
    pos_scores = (q_proj * p_proj).sum(-1) + lex_bonus
    neg_scores = (q_proj * n_proj).sum(-1) + model.lex_score(q_proj, neg_lex)
    pair_loss  = torch.clamp(1.0 - pos_scores + neg_scores, min=0).mean()

    return ce_loss + 0.2 * pair_loss

# --- load data ---
print("Loading MS MARCO v2.1...")
msmarco = load_dataset("microsoft/ms_marco", "v2.1", split="train")
print(f"Loaded {len(msmarco)} queries")

# --- tokenizer ---
backbone  = "./bert-wiki-backbone" if os.path.exists("./bert-wiki-backbone") else "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(backbone)

# --- load model from NQ checkpoint ---
model = LexBertRanker(bert_model_name=backbone, n_lex_features=5)

if os.path.exists("lex_bert_nq.pt"):
    print("Loading NQ checkpoint...")
    state = torch.load("lex_bert_nq.pt", map_location="cpu")

    # load BERT backbone weights only — skip q_proj, p_proj, lex heads
    # because those were trained with /8.0 scaling and will cause collapsed loss
    bert_state = {k: v for k, v in state.items() if k.startswith("bert.")}
    missing, unexpected = model.load_state_dict(bert_state, strict=False)
    print(f"Loaded BERT backbone — {len(bert_state)} keys")
    print(f"Freshly initialized: {[k for k in missing]}")

    # reinitialize projection heads cleanly
    nn.init.xavier_uniform_(model.q_proj.weight)
    nn.init.zeros_(model.q_proj.bias)
    nn.init.xavier_uniform_(model.p_proj.weight)
    nn.init.zeros_(model.p_proj.bias)
    nn.init.xavier_uniform_(model.lex_projection.weight)
    nn.init.zeros_(model.lex_projection.bias)
    nn.init.xavier_uniform_(model.lex_classifier.weight)
    nn.init.zeros_(model.lex_classifier.bias)
    print("Projection heads reinitialized with Xavier uniform")
else:
    print("WARNING: no NQ checkpoint found, training from backbone")

model = model.to(device)

# --- optimizer ---
optimizer = optim.AdamW([
    {"params": model.bert.parameters(),           "lr": 5e-5},
    {"params": model.q_proj.parameters(),         "lr": 3e-4},
    {"params": model.p_proj.parameters(),         "lr": 3e-4},
    {"params": model.lex_projection.parameters(), "lr": 3e-4},
    {"params": model.lex_classifier.parameters(), "lr": 3e-4},
], weight_decay=0.01)

# --- dataset and loader ---
dataset = MarcoRankingDataset(msmarco, tokenizer, max_examples=200000)
loader  = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=0)

# --- scheduler ---
total_steps  = len(loader)
warmup_steps = total_steps // 10
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# --- training loop ---
print(f"\nStarting Stage 4 MS MARCO fine-tuning...")
print(f"Steps: {len(loader)} | Warmup: {warmup_steps} steps\n")

model.train()
total_loss = 0
best_loss  = float("inf")

for step, batch in enumerate(loader):
    optimizer.zero_grad()

    loss = ranking_loss(model, batch, device, temperature=0.05)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()

    total_loss += loss.item()

    if step % 100 == 0:
        avg = total_loss / (step + 1)
        lr  = scheduler.get_last_lr()[0]
        print(f"  Step {step}/{len(loader)} | Loss: {loss.item():.4f} | Avg: {avg:.4f} | LR: {lr:.2e}")

    # save best checkpoint
    if loss.item() < best_loss:
        best_loss = loss.item()
        torch.save(model.state_dict(), "lex_bert_marco_best.pt")

    # save every 500 steps
    if step % 500 == 0 and step > 0:
        ckpt_path = f"lex_bert_marco_step{step}.pt"
        torch.save(model.state_dict(), ckpt_path)
        print(f"  ↳ Checkpoint saved: {ckpt_path}")

avg_loss = total_loss / len(loader)
print(f"\nStage 4 complete | Avg loss: {avg_loss:.4f}")

torch.save(model.state_dict(), "lex_bert_marco_final.pt")
print("Saved: lex_bert_marco_final.pt")
print("Saved: lex_bert_marco_best.pt  (best step checkpoint)")

GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Memory free: 3.95 GB
Loading MS MARCO v2.1...
Loaded 808731 queries


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 11371.53it/s]
[transformers] BertModel LOAD REPORT from: ./bert-wiki-backbone
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading NQ checkpoint...
Loaded BERT backbone — 199 keys
Freshly initialized: ['lex_projection.weight', 'lex_projection.bias', 'q_proj.weight', 'q_proj.bias', 'p_proj.weight', 'p_proj.bias', 'lex_classifier.weight', 'lex_classifier.bias']
Projection heads reinitialized with Xavier uniform
Building pairwise triplets from MS MARCO...
Built 200000 triplets | Skipped 39901 queries with no pos/neg

Starting Stage 4 MS MARCO fine-tuning...
Steps: 12500 | Warmup: 1250 steps

  Step 0/12500 | Loss: 5.9089 | Avg: 5.9089 | LR: 4.00e-08
  Step 100/12500 | Loss: 0.1999 | Avg: 2.0463 | LR: 4.04e-06
  Step 200/12500 | Loss: 0.1988 | Avg: 1.1264 | LR: 8.04e-06
  Step 300/12500 | Loss: 0.1861 | Avg: 0.8127 | LR: 1.20e-05
  Step 400/12500 | Loss: 0.1444 | Avg: 0.6521 | LR: 1.60e-05
  Step 500/12500 | Loss: 0.1807 | Avg: 0.5554 | LR: 2.00e-05
  ↳ Checkpoint saved: lex_bert_marco_step500.pt
  Step 600/12500 | Loss: 0.1601 | Avg: 0.4877 | LR: 2.40e-05
  Step 700/12500 | Loss: 0.1448 | Avg: 0.4402 | LR: 2.

In [ ]:
import gc
import os
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer, BertModel
import subprocess
subprocess.run(["pip", "install", "matplotlib"], check=True)
import matplotlib.pyplot as plt
from collections import defaultdict

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

# --- LexBERT (must match Stage 4 architecture) ---
class LexBertRanker(nn.Module):
    def __init__(self, bert_model_name="bert-base-uncased", n_lex_features=5):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        hidden = self.bert.config.hidden_size

        self.lex_projection = nn.Linear(n_lex_features, 64)
        self.relu           = nn.ReLU()
        self.dropout        = nn.Dropout(0.1)
        self.q_proj         = nn.Linear(hidden, 256)
        self.p_proj         = nn.Linear(hidden, 256)
        self.lex_classifier = nn.Linear(256 + 64, 1)

    def encode(self, input_ids, attention_mask, token_type_ids):
        out = self.bert(input_ids=input_ids,
                        attention_mask=attention_mask,
                        token_type_ids=token_type_ids)
        return out.last_hidden_state[:, 0, :]

    def project_query(self, cls_vec):
        return self.q_proj(cls_vec)   # no /8.0 — Stage 4 version

    def project_passage(self, cls_vec):
        return self.p_proj(cls_vec)

    def lex_score(self, q_proj, lex_features):
        lex_vec  = self.dropout(self.relu(self.lex_projection(lex_features)))
        combined = torch.cat([q_proj, lex_vec], dim=1)
        return self.lex_classifier(combined).squeeze(-1)

    def forward(self, q_input_ids, q_attention_mask, q_token_type_ids,
                p_input_ids, p_attention_mask, p_token_type_ids, lex_features):
        q_cls  = self.encode(q_input_ids, q_attention_mask, q_token_type_ids)
        p_cls  = self.encode(p_input_ids, p_attention_mask, p_token_type_ids)
        q_proj = torch.nn.functional.normalize(self.project_query(q_cls), dim=-1)
        p_proj = torch.nn.functional.normalize(self.project_passage(p_cls), dim=-1)
        dot    = (q_proj * p_proj).sum(dim=-1)
        lex    = self.lex_score(q_proj, lex_features)
        return dot + lex

# --- Base BERT scorer ---
class BaseBertScorer(nn.Module):
    def __init__(self, bert_model_name="bert-base-uncased"):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)

    def forward(self, q_ids, q_mask, q_type, p_ids, p_mask, p_type):
        q_cls = self.bert(input_ids=q_ids, attention_mask=q_mask,
                          token_type_ids=q_type).last_hidden_state[:, 0, :]
        p_cls = self.bert(input_ids=p_ids, attention_mask=p_mask,
                          token_type_ids=p_type).last_hidden_state[:, 0, :]
        q_cls = torch.nn.functional.normalize(q_cls, dim=-1)
        p_cls = torch.nn.functional.normalize(p_cls, dim=-1)
        return (q_cls * p_cls).sum(dim=-1)

# --- lexical features ---
def compute_lex_features(query, passage):
    qt = query.lower().split()
    pt = passage.lower().split()
    qs, ps = set(qt), set(pt)
    overlap  = len(qs & ps) / len(qs) if qs else 0.0
    jaccard  = len(qs & ps) / len(qs | ps) if (qs | ps) else 0.0
    try:
        mat = TfidfVectorizer().fit_transform([query, passage])
        tfidf_sim = cosine_similarity(mat[0], mat[1])[0][0]
    except:
        tfidf_sim = 0.0
    length  = np.log1p(len(pt)) / 10.0
    density = sum(pt.count(t) for t in qt) / max(len(pt), 1)
    return np.array([overlap, jaccard, tfidf_sim, length, density], dtype=np.float32)

# --- metrics ---
def mean_reciprocal_rank(ranked_labels):
    for rank, label in enumerate(ranked_labels, start=1):
        if label == 1:
            return 1.0 / rank
    return 0.0

def recall_at_k(ranked_labels, k):
    total_pos = sum(ranked_labels)
    if total_pos == 0:
        return 0.0
    return sum(ranked_labels[:k]) / total_pos

def ndcg_at_k(ranked_labels, k):
    dcg  = sum(l / np.log2(r + 2) for r, l in enumerate(ranked_labels[:k]))
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(sum(ranked_labels), k)))
    return dcg / idcg if idcg > 0 else 0.0

def precision_at_k(ranked_labels, k):
    return sum(ranked_labels[:k]) / k

# --- score all passages for one query, return raw scores ---
@torch.no_grad()
def score_passages(model, tokenizer, query, passages, device, is_lexbert=True):
    scores = []
    for passage in passages:
        qe = tokenizer(query,   max_length=64,  padding="max_length",
                       truncation=True, return_tensors="pt")
        pe = tokenizer(passage, max_length=128, padding="max_length",
                       truncation=True, return_tensors="pt")

        q_ids  = qe["input_ids"].to(device)
        q_mask = qe["attention_mask"].to(device)
        q_type = qe["token_type_ids"].to(device)
        p_ids  = pe["input_ids"].to(device)
        p_mask = pe["attention_mask"].to(device)
        p_type = pe["token_type_ids"].to(device)

        if is_lexbert:
            lex = torch.tensor(compute_lex_features(query, passage)).unsqueeze(0).to(device)
            s   = model(q_ids, q_mask, q_type, p_ids, p_mask, p_type, lex)
        else:
            s = model(q_ids, q_mask, q_type, p_ids, p_mask, p_type)

        scores.append(s.item())
    return scores

# --- load MS MARCO validation ---
print("Loading MS MARCO v2.1 validation split...")
msmarco = load_dataset("microsoft/ms_marco", "v2.1", split="validation")
print(f"Total validation queries: {len(msmarco)}")

# filter to queries with >4 passages and at least 1 positive
print("Filtering queries with >4 passages and at least 1 positive...")
eval_queries = []
for ex in msmarco:
    texts    = ex["passages"]["passage_text"]
    selected = ex["passages"]["is_selected"]
    if len(texts) > 4 and sum(selected) >= 1:
        eval_queries.append({
            "query":    ex["query"],
            "passages": texts,
            "labels":   selected,
            "n_pos":    sum(selected),
            "n_total":  len(texts),
        })

print(f"Found {len(eval_queries)} qualifying queries")
print(f"Avg passages per query: {np.mean([q['n_total'] for q in eval_queries]):.1f}")
print(f"Avg positives per query: {np.mean([q['n_pos'] for q in eval_queries]):.1f}")

# --- load tokenizer ---
backbone  = "./bert-wiki-backbone" if os.path.exists("./bert-wiki-backbone") else "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(backbone)

# --- load Base BERT ---
print("\nLoading Base BERT...")
base_model = BaseBertScorer(bert_model_name="bert-base-uncased").to(device)
base_model.eval()

# --- load LexBERT ---
print("Loading LexBERT...")
lex_model = LexBertRanker(bert_model_name=backbone, n_lex_features=5)
for ckpt in ["lex_bert_marco_final.pt", "lex_bert_marco_best.pt", "lex_bert_nq.pt"]:
    if os.path.exists(ckpt):
        print(f"  Using checkpoint: {ckpt}")
        lex_model.load_state_dict(torch.load(ckpt, map_location="cpu"))
        lex_ckpt_name = ckpt
        break
lex_model = lex_model.to(device)
lex_model.eval()

# --- evaluation ---
print(f"\nEvaluating {len(eval_queries)} queries...")
K_VALUES = [1, 3, 5, 10]

base_results = defaultdict(list)
lex_results  = defaultdict(list)
per_query    = []

for i, ex in enumerate(eval_queries):
    query    = ex["query"]
    passages = ex["passages"]
    labels   = ex["labels"]

    # get raw relevance scores from both models
    base_scores = score_passages(base_model, tokenizer, query, passages, device, is_lexbert=False)
    lex_scores  = score_passages(lex_model,  tokenizer, query, passages, device, is_lexbert=True)

    # rank passages by score descending
    base_order  = np.argsort(base_scores)[::-1]
    lex_order   = np.argsort(lex_scores)[::-1]

    base_ranked = [labels[j] for j in base_order]
    lex_ranked  = [labels[j] for j in lex_order]

    # metrics
    base_mrr = mean_reciprocal_rank(base_ranked)
    lex_mrr  = mean_reciprocal_rank(lex_ranked)

    base_results["mrr"].append(base_mrr)
    lex_results["mrr"].append(lex_mrr)

    for k in K_VALUES:
        base_results[f"recall@{k}"].append(recall_at_k(base_ranked, k))
        lex_results[f"recall@{k}"].append(recall_at_k(lex_ranked, k))
        base_results[f"ndcg@{k}"].append(ndcg_at_k(base_ranked, k))
        lex_results[f"ndcg@{k}"].append(ndcg_at_k(lex_ranked, k))
        base_results[f"p@{k}"].append(precision_at_k(base_ranked, k))
        lex_results[f"p@{k}"].append(precision_at_k(lex_ranked, k))

    per_query.append({
        "query":        query,
        "n_passages":   ex["n_total"],
        "n_pos":        ex["n_pos"],
        "labels":       labels,
        "base_scores":  base_scores,
        "lex_scores":   lex_scores,
        "base_order":   base_order.tolist(),
        "lex_order":    lex_order.tolist(),
        "base_ranked":  base_ranked,
        "lex_ranked":   lex_ranked,
        "base_mrr":     base_mrr,
        "lex_mrr":      lex_mrr,
        "mrr_delta":    lex_mrr - base_mrr,
    })

    if (i + 1) % 100 == 0:
        print(f"  [{i+1}/{len(eval_queries)}] "
              f"Base MRR: {np.mean(base_results['mrr']):.4f} | "
              f"Lex MRR:  {np.mean(lex_results['mrr']):.4f}")

# --- results table ---
print("\n" + "═" * 65)
print(f"  RESULTS — {len(eval_queries)} queries | checkpoint: {lex_ckpt_name}")
print("═" * 65)
print(f"{'Metric':<20} {'Base BERT':>12} {'LexBERT':>12} {'Delta':>10}")
print("─" * 65)

for m in (["mrr"] +
          [f"ndcg@{k}"   for k in K_VALUES] +
          [f"recall@{k}" for k in K_VALUES] +
          [f"p@{k}"      for k in K_VALUES]):
    bv    = np.mean(base_results[m])
    lv    = np.mean(lex_results[m])
    delta = lv - bv
    arrow = "▲" if delta > 0 else "▼"
    print(f"{m:<20} {bv:>12.4f} {lv:>12.4f} {arrow}{abs(delta):>9.4f}")

print("═" * 65)

# --- per-query win/loss breakdown ---
deltas    = [q["mrr_delta"] for q in per_query]
lex_wins  = sum(1 for d in deltas if d > 0)
lex_loses = sum(1 for d in deltas if d < 0)
ties      = sum(1 for d in deltas if d == 0)
print(f"\nPer-query MRR:  LexBERT wins {lex_wins} ({100*lex_wins/len(deltas):.1f}%) | "
      f"loses {lex_loses} ({100*lex_loses/len(deltas):.1f}%) | "
      f"ties {ties} ({100*ties/len(deltas):.1f}%)")

# --- show 5 best and worst queries ---
print("\nTop 5 queries where LexBERT improved most:")
for q in sorted(per_query, key=lambda x: x["mrr_delta"], reverse=True)[:5]:
    print(f"  +{q['mrr_delta']:.3f} | Base {q['base_mrr']:.3f} → Lex {q['lex_mrr']:.3f} | {q['query'][:70]}")

print("\nTop 5 queries where LexBERT hurt most:")
for q in sorted(per_query, key=lambda x: x["mrr_delta"])[:5]:
    print(f"  {q['mrr_delta']:.3f} | Base {q['base_mrr']:.3f} → Lex {q['lex_mrr']:.3f} | {q['query'][:70]}")

# --- show passage-level scores for a sample query ---
print("\n--- Sample query passage scores ---")
sample = per_query[0]
print(f"Query: {sample['query']}")
print(f"{'Rank':<6} {'Base Score':>12} {'Lex Score':>12} {'Label':>8} {'Passage[:60]'}")
print("─" * 95)

# show passages sorted by LexBERT score
lex_sorted = sorted(
    enumerate(zip(sample["base_scores"], sample["lex_scores"], sample["labels"],
                  eval_queries[0]["passages"])),
    key=lambda x: x[1][1], reverse=True
)
for rank, (orig_idx, (bs, ls, label, passage)) in enumerate(lex_sorted, 1):
    marker = "✓" if label == 1 else " "
    print(f"  {rank:<4} {bs:>12.4f} {ls:>12.4f} {marker:>8}   {passage[:60]}...")

# --- plots ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f"Base BERT vs LexBERT — MS MARCO ({len(eval_queries)} queries)\n"
             f"Checkpoint: {lex_ckpt_name}", fontsize=13)

# 1. MRR distribution
ax = axes[0, 0]
ax.hist(base_results["mrr"], bins=30, alpha=0.6, label="Base BERT", color="steelblue")
ax.hist(lex_results["mrr"],  bins=30, alpha=0.6, label="LexBERT",   color="coral")
ax.axvline(np.mean(base_results["mrr"]), color="steelblue", linestyle="--", lw=2,
           label=f"Base mean={np.mean(base_results['mrr']):.3f}")
ax.axvline(np.mean(lex_results["mrr"]),  color="coral",     linestyle="--", lw=2,
           label=f"Lex mean={np.mean(lex_results['mrr']):.3f}")
ax.set_title("MRR Distribution")
ax.set_xlabel("MRR")
ax.set_ylabel("# Queries")
ax.legend(fontsize=8)

# 2. NDCG@K
ax = axes[0, 1]
x, w = np.arange(len(K_VALUES)), 0.35
ax.bar(x - w/2, [np.mean(base_results[f"ndcg@{k}"]) for k in K_VALUES],
       w, label="Base BERT", color="steelblue", alpha=0.8)
ax.bar(x + w/2, [np.mean(lex_results[f"ndcg@{k}"]) for k in K_VALUES],
       w, label="LexBERT",   color="coral",     alpha=0.8)
ax.set_title("NDCG@K")
ax.set_xticks(x)
ax.set_xticklabels([f"@{k}" for k in K_VALUES])
ax.set_ylabel("NDCG")
ax.legend()

# 3. Recall@K
ax = axes[0, 2]
ax.bar(x - w/2, [np.mean(base_results[f"recall@{k}"]) for k in K_VALUES],
       w, label="Base BERT", color="steelblue", alpha=0.8)
ax.bar(x + w/2, [np.mean(lex_results[f"recall@{k}"]) for k in K_VALUES],
       w, label="LexBERT",   color="coral",     alpha=0.8)
ax.set_title("Recall@K")
ax.set_xticks(x)
ax.set_xticklabels([f"@{k}" for k in K_VALUES])
ax.set_ylabel("Recall")
ax.legend()

# 4. Per-query MRR scatter
ax = axes[1, 0]
ax.scatter([q["base_mrr"] for q in per_query],
           [q["lex_mrr"]  for q in per_query],
           alpha=0.3, s=12, color="purple")
lim = max(max(base_results["mrr"]), max(lex_results["mrr"])) + 0.05
ax.plot([0, lim], [0, lim], "k--", lw=1, label="No change")
ax.set_title("Per-query MRR\n(above diagonal = LexBERT wins)")
ax.set_xlabel("Base BERT MRR")
ax.set_ylabel("LexBERT MRR")
ax.legend()

# 5. MRR delta histogram
ax = axes[1, 1]
ax.hist(deltas, bins=40, color="purple", alpha=0.7)
ax.axvline(0, color="black", linestyle="--", lw=1)
ax.axvline(np.mean(deltas), color="red", linestyle="--", lw=2,
           label=f"Mean delta={np.mean(deltas):.4f}")
ax.set_title("Per-query MRR Delta\n(LexBERT − Base BERT)")
ax.set_xlabel("MRR Delta")
ax.set_ylabel("# Queries")
ax.legend()

# 6. Score distribution for a sample query
ax = axes[1, 2]
sample      = per_query[0]
n_passages  = len(sample["base_scores"])
passage_ids = np.arange(n_passages)
colors      = ["green" if l == 1 else "gray" for l in sample["labels"]]

# sort by lex score for cleaner visualization
sort_idx    = np.argsort(sample["lex_scores"])[::-1]
base_sorted = [sample["base_scores"][i] for i in sort_idx]
lex_sorted_scores  = [sample["lex_scores"][i]  for i in sort_idx]
label_sorted = [sample["labels"][i] for i in sort_idx]
bar_colors  = ["green" if l == 1 else "gray" for l in label_sorted]

ax.bar(passage_ids - 0.2, base_sorted,      0.4, label="Base BERT", color="steelblue", alpha=0.7)
ax.bar(passage_ids + 0.2, lex_sorted_scores, 0.4, label="LexBERT",   color="coral",     alpha=0.7)
for idx, color in enumerate(bar_colors):
    if color == "green":
        ax.axvline(idx, color="green", alpha=0.3, lw=8)
ax.set_title(f"Passage Scores — Sample Query\n\"{sample['query'][:50]}...\"")
ax.set_xlabel("Passage rank (by LexBERT score)")
ax.set_ylabel("Relevance score")
ax.legend()

plt.tight_layout()
plt.savefig("evaluation_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nSaved: evaluation_results.png")

ModuleNotFoundError: No module named 'matplotlib'